### 模型初始化与自动下载

在进行多模态推理和图像生成之前，先将视觉大模型（Qwen3-VL）与扩散生成模型（Z-Image-Turbo）加载至内存中。

In [1]:
from pathlib import Path
from notebook_utils import device_widget
from optimum.intel.openvino import OVModelForVisualCausalLM
from optimum.intel import OVZImagePipeline
from modelscope import snapshot_download  # 新增的下载库

device = device_widget(default="AUTO", exclude=["NPU"])

# ==========================================
# 1. 检查并加载 Qwen3-VL 模型
# ==========================================
qwen_vl_model_dir = Path(r"./Qwen3-VL-4B-Instruct-ov-int4")

if not qwen_vl_model_dir.exists():
    print(f"⏳ 目录不存在，正在从 ModelScope 下载 Qwen3-VL 模型到 {qwen_vl_model_dir}...")
    snapshot_download("snake7gun/Qwen3-VL-4B-Instruct-int4-ov", local_dir=str(qwen_vl_model_dir))
else:
    print(f"📦 Qwen3-VL 模型已存在，跳过下载。")

model = OVModelForVisualCausalLM.from_pretrained(qwen_vl_model_dir, device=device.value)
print("✅ Qwen3-VL-4B 模型加载完成")

# ==========================================
# 2. 检查并加载 Z-Image-Turbo 模型
# ==========================================
z_image_model_dir = Path(r"./Z-Image-Turbo-int4-ov")

if not z_image_model_dir.exists():
    print(f"⏳ 目录不存在，正在从 ModelScope 下载 Z-Image-Turbo 模型到 {z_image_model_dir}...")
    snapshot_download("snake7gun/Z-Image-Turbo-int4-ov", local_dir=str(z_image_model_dir))
else:
    print(f"📦 Z-Image-Turbo 模型已存在，跳过下载。")

ov_pipe = OVZImagePipeline.from_pretrained(z_image_model_dir, device=device.value)
print("✅ Z-Image-Turbo 模型加载完成")

Multiple distributions found for package optimum. Picked distribution: optimum
D:\Software\anaconda3\envs\openvino\Lib\site-packages\torch\amp\autocast_mode.py:270: UserWarning: User provided device_type of 'cuda', but CUDA is not available. Disabling
  warnings.warn(


✅ Qwen3-VL-4B 模型加载完成
✅ Z-Image-Turbo 模型加载完成


### 构建相关处理器

In [2]:
from PIL import Image
from transformers import AutoProcessor, TextStreamer
from pathlib import Path
import requests

min_pixels = 256 * 28 * 28
max_pixels = 1280 * 28 * 28
processor = AutoProcessor.from_pretrained(qwen_vl_model_dir, min_pixels=min_pixels, max_pixels=max_pixels)

### 构建界面

In [3]:
import gradio as gr
import torch
from PIL import Image

# ==========================================
# 核心逻辑：单一、感知的智能工作流 (Jupyter 专属)
# ==========================================

def smart_unified_workflow(sketch_dict, text_unified_input, photo_edit_content, photo_transfer_content, photo_transfer_style, style_input_global, workflow_mode):
    """
    统一创意中枢，智能感知你的创作场景。
    workflow_mode: gen_mode (Tab 1), photo_edit_mode (Tab 2), style_transfer_mode (Tab 3)
    """
    text_intent = text_unified_input.strip() if text_unified_input else ""
    messages = [{"role": "user", "content": []}] # 定义统一的多模态消息结构
    final_vlm_query = ""

    # --- 场景一：灵感共创 (手绘手绘+语言 或 纯语言) ---
    if workflow_mode == "gen_mode":
        sketch_img_pil = sketch_dict["composite"] if (sketch_dict is not None and "composite" in sketch_dict) else None
        
        # 智能检测：画板是不是空的？
        is_sketch_empty = True
        if sketch_img_pil is not None:
            # PIL getcolors 快速检查是否只有一种颜色（通常是白色）
            colors = sketch_img_pil.getcolors()
            if colors and (len(colors) > 1 or colors[0][1] != (255, 255, 255)):
                is_sketch_empty = False
        
        if not text_intent and is_sketch_empty:
            yield None, "亲，哪怕随便给一个简单的想法（模式一），AI 也能帮你脑补出来哦！"
            return

        # 根据画板状态智能路由
        if not is_sketch_empty:
            print("💡 智能识别场景一：手绘草图 + 语言定制。正在融合构图...")
            final_vlm_query = f"你是一个多模态风格翻译助手。请仔细观察这张草图轮廓，定下物理构图。然后融合用户想法：【{text_intent or '自由发挥'}】，扩写成一段高质量提示词。\n【极其重要的风格强制要求】：{style_input_global or '微信表情包风格'}。请直接输出最终的提示词文本，绝对不要包含废话。"
            messages[0]["content"].insert(0, {"type": "image", "image": sketch_img_pil})
        else:
            print("💡 智能识别场景一：极简文字驱动。正在脑补DETAILS...")
            final_vlm_query = f"你是一个多模态扩写助手。请把极简的想法：'{text_intent}'，扩写成一段画面感极强、细节丰富的图像生成提示词。\n【极其重要的风格强制要求】：{style_input_global or '微信表情包风格'}。请直接输出最终的提示词文本，绝不要废话。"
        
    # --- 场景二 (就是你问的这个！)：上传图片 + 语言修改/风格定制 ---
    elif workflow_mode == "photo_edit_mode":
        if photo_edit_content is None:
            yield None, "亲，场景二需要上传一张你想修改的照片！", "" # 占位
            return
        
        print(f"🎨 智能识别场景二：视觉语义修改与深度定制。正在对原照片进行重塑...")
        # 此场景需要 VLM 理解原图语义，然后根据文字进行修改/风格转换
        final_vlm_query = f"""你是一个多模态图像重塑专家。请观察这张照片，提取主体（长相衣着）、动作和环境。
        你的任务是：在【保持原主体 ID、动作姿态不变】的前提下，忽略原图的写实质感，根据用户的深度定制要求进行全景重塑。
        【用户的深度定制/修改要求】：{text_intent or '保持原意'}。\n【极其重要的强制新风格要求】：必须将整体画面适配以下风格：{style_input_global or '保持原风格'}。请直接输出提示词文本，严禁废话。"""
        
        messages[0]["content"].insert(0, {"type": "image", "image": photo_edit_content})

    # --- 场景三：上传图片上传图片 + 上传风格图片上传风格图片 ---
    elif workflow_mode == "style_transfer_mode":
        if photo_transfer_content is None or photo_transfer_style is None:
            yield None, "亲，场景三需要上传内容参考照片 AND 风格参考照片！" 
            return
        
        print(f"🎨 智能识别场景三：纯语义风格参考重绘。正在融合风格和内容语义...")
        # 针对Qwen-VL对多图支持的适配：提取内容图语义
        final_vlm_query = f"""你是一个顶级多模态风格迁移专家。
        【输入一（内容参考）】：是一张需要重绘的照片。
        【输入二（风格参考）】：是一张目标艺术风格的样例图。
        要求：请仔细认出内容图里的核心主体（长相、衣着、动作）和场景，彻底忽略原有的写实质感，完全提取语义细节。同时，请提炼风格图里的核心艺术风格（材质、光影、色调、艺术流派）。最后，重新撰写一段高质量图像生成提示词，将【内容参考】的语义完全适配到【风格参考】的艺术风格里。\n【用户附加文字】：{text_intent or '保持原意'}。请直接输出提示词文本，绝对不要废话。"""
        
        # 将内容图作为 Image 输入，风格描述作为 adapter 手动调用 VLM (工程适配)
        messages[0]["content"].insert(0, {"type": "image", "image": photo_transfer_content})

    # 添加文字指令到消息列表
    messages[0]["content"].append({"type": "text", "text": final_vlm_query})

    # --- A. 呼叫 Qwen-VL 生成提示词 (本地推理) ---
    print("\n🧠 Qwen-VL 正在进行语义解析与脑补...")
    inputs = processor.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_dict=True, return_tensors="pt")
    generated_ids = model.generate(**inputs, max_new_tokens=300)
    generated_ids_trimmed = [out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)]
    
    final_vlm_prompt = processor.batch_decode(generated_ids_trimmed, skip_special_tokens=True)[0].strip()
    # 清理非纯文本字符，防止报错形状不齐
    final_prompt = final_vlm_prompt.replace("\n", "").replace("*", "").strip()

    # 🌟 提前输出提示词：让用户马上看到 AI 的“思考过程”
    yield None, final_vlm_prompt 

    # --- B. 呼叫 Z-Image 渲染图像 (本地生成) ---
    print("\n🎨 提示词已就绪，正在驱动 Z-Image 渲染图像...")
    negative_prompt = "丑陋, 变形, 模糊, 糟糕的解剖结构, 极简, 缺维, 乱码, 水印, 文字, 签名, 低画质, 毁容"
    
    try:
        final_image = ov_pipe(
            prompt=final_prompt,
            negative_prompt=negative_prompt,
            height=1024,
            width=1024,
            num_inference_steps=9,     # 保持原始代码步数
            guidance_scale=0.0,        # 保持原始代码 scale
            generator=torch.Generator("cpu").manual_seed(torch.randint(0, 1000000, (1,)).item()), 
        ).images[0]
        
        yield final_image, final_vlm_prompt
        
    except Exception as e:
        # 如果报错长度不齐，清理可能存在的换行符再试一次
        cleaned_prompt = final_prompt.replace("\n", "").strip()
        print(f"ZImage 生成报错，尝试清理 Prompt 后重试: {str(e)}")
        yield None, f"图像生成失败，底层报错：{str(e)}"

# ==========================================
# Gradio 极简开放式界面构建 (Jupyter 专属)
# ==========================================

try:
    demo_app.close()
except:
    pass

with gr.Blocks(title="全能创意工坊", theme=gr.themes.Soft()) as demo_app:
    gr.Markdown(
        """
        # 🚀 个人专属数字创意兵工厂
        无需学习复杂的提示词工程，选一个你喜欢的方式，立刻把脑洞变成现实！
        """
    )
    
    with gr.Tabs():
        
        # --------------------------------------------------
        # 场景一：灵感共创 (手绘+语言 或 纯语言)
        # --------------------------------------------------
        with gr.Tab("🎨 灵感共创 (手绘+语言 或 纯语言)"):
            gr.Markdown("### 场景一：只写一句大白话秒出图，或者结合你手绘的构图线条。哪怕空着画板也行！AI 都懂你。")
            with gr.Row():
                with gr.Column(scale=1):
                    text_unified_input_gen = gr.Textbox(
                        label="想画点什么？", 
                        placeholder="伤心打工人 / 赛博小猫 / 不想工作...", 
                        lines=2
                    )
                    
                    style_input_global_gen = gr.Textbox(
                        label="兜底风格许愿池 (可选)", 
                        placeholder="例如：3D泡泡玛特盲盒 / 中国传统写意水墨画 / 赛博朋克霓虹风...", 
                        lines=1
                    )
                    gr.Examples(examples=["3D泡泡玛特盲盒，辛烷值渲染", "中国传统写意水墨画，留白意境", "赛博朋克霓虹风"], inputs=style_input_global_gen, label="💡 没灵感？点击预设风格填入")

                    canvas_unified_input_gen = gr.Sketchpad(
                        label="手绘画板 (可选构图线条)", 
                        type="pil", 
                        height=400,
                        brush=gr.Brush(default_size=3, colors=["#000000"]) 
                    )
                    
                    submit_btn_gen = gr.Button("✨ 生成创意大片", variant="primary", size="lg")
                
                with gr.Column(scale=1):
                    output_image_gen = gr.Image(label="为你生成的专属创意大片", type="pil", height=400)
                    output_prompt_gen = gr.Textbox(label="AI 脑补的完整指令 (实时更新)", interactive=False, lines=6)

            def adapter_mode_gen(sketch, text, style):
                gen = smart_unified_workflow(sketch, text, None, None, None, style, "gen_mode")
                for outputs in gen:
                    yield outputs[0], outputs[1]

            submit_btn_gen.click(
                fn=adapter_mode_gen,
                inputs=[canvas_unified_input_gen, text_unified_input_gen, style_input_global_gen],
                outputs=[output_image_gen, output_prompt_gen]
            )

        # --------------------------------------------------
        # 场景二：照片语义修改/风格定制
        # --------------------------------------------------
        with gr.Tab("🖼️ 照片深度定制 (上传原图+语言修改)"):
            gr.Markdown("### 场景二：上传一张内容参考照片，用大白话提出修改意见（加东西、换画风），系统在原图语义基础上重新创作。")
            with gr.Row():
                with gr.Column(scale=1):
                    input_photo_edit_content = gr.Image(label="上传你想修改的原图", type="pil", height=400)
                    
                    text_edit_hint = gr.Textbox(
                        label="内容修改意见 (大白话)", 
                        placeholder="例如：把他变成黄色 / 让他戴墨镜度假 / 变得愤怒一点...", 
                        lines=2
                    )
                    
                    style_input_global_edit = gr.Textbox(
                        label="目标新画风许愿池 (可选)", 
                        placeholder="例如：3D泡泡玛特盲盒 / 二次元日系动漫", 
                        lines=1
                    )
                    
                    submit_btn_edit = gr.Button("🪄 一键智能重塑", variant="primary", size="lg")
                
                with gr.Column(scale=1):
                    output_image_edit = gr.Image(label="重塑后的专属大片", type="pil", height=400)
                    output_prompt_edit = gr.Textbox(label="AI 为照片语义重塑生成的指令", interactive=False, lines=6)
            
            def adapter_mode_photo_edit(photo, hint, style_global):
                gen = smart_unified_workflow(None, hint, photo, None, None, style_global, "photo_edit_mode")
                for outputs in gen:
                    yield outputs[0], outputs[1]

            submit_btn_edit.click(
                fn=adapter_mode_photo_edit,
                inputs=[input_photo_edit_content, text_edit_hint, style_input_global_edit],
                outputs=[output_image_edit, output_prompt_edit]
            )

        # --------------------------------------------------
        # 场景三：照片语义转换 (上传内容图 AND 上传风格参考图)
        # --------------------------------------------------
        with gr.Tab("🖼️ 风格参考重塑 (上传照片+上传风格图)"):
            gr.Markdown("### 场景三：上传一张内容图 A，上传一张风格图 B。AI 会提取图 A 的语义，并在图 B 的画风里重新创作！")
            with gr.Row():
                with gr.Column(scale=1):
                    input_content_photo_transfer = gr.Image(label="上传内容参考图片 (提取人物/动作)", type="pil", height=350)
                    input_style_photo_transfer = gr.Image(label="上传风格参考图片 (提取画风/色调)", type="pil", height=350)
                    
                    text_transfer_hint = gr.Textbox(
                        label="补充文字定制意见 (可选)", 
                        placeholder="例如：改成黄色、给他加把雨伞...", 
                        lines=1
                    )
                    
                    submit_btn_transfer = gr.Button("✨ 一键风格魔法", variant="primary", size="lg")
                
                with gr.Column(scale=1):
                    output_image_transfer = gr.Image(label="生成的精美大片", type="pil", height=400)
                    output_prompt_transfer = gr.Textbox(label="AI 为照片脑补的风格重塑指令", interactive=False, lines=6)
            
            def adapter_mode_photo_transfer(photo, style_ref, hint):
                # 场景三不需要全局文字风格输入，传入 None 即可
                gen = smart_unified_workflow(None, hint, None, photo, style_ref, None, "style_transfer_mode")
                for outputs in gen:
                    yield outputs[0], outputs[1]

            submit_btn_transfer.click(
                fn=adapter_mode_photo_transfer,
                inputs=[input_content_photo_transfer, input_style_photo_transfer, text_transfer_hint],
                outputs=[output_image_transfer, output_prompt_transfer]
            )

demo_app.launch(inline=True, height=950)

C:\Users\wuzm\AppData\Local\Temp\ipykernel_1868\236103357.py:121: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(title="全能创意工坊", theme=gr.themes.Soft()) as demo_app:


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


💡 智能识别场景一：手绘草图 + 语言定制。正在融合构图...

🧠 Qwen-VL 正在进行语义解析与脑补...

🎨 提示词已就绪，正在驱动 Z-Image 渲染图像...


  0%|          | 0/9 [00:00<?, ?it/s]

💡 智能识别场景一：手绘草图 + 语言定制。正在融合构图...

🧠 Qwen-VL 正在进行语义解析与脑补...

🎨 提示词已就绪，正在驱动 Z-Image 渲染图像...


  0%|          | 0/9 [00:00<?, ?it/s]